# Incremental GNN for Dynamic Graph Coloring

This notebook is the current reproducible implementation used for the project.

The evaluation now:
- checks **every undirected edge** for conflicts;
- repairs residual GNN conflicts until the coloring is valid;
- uses a real simulated-annealing baseline with temperature cooling and Metropolis acceptance;
- evaluates on **held-out graphs** that are not used for training;
- applies the stated **70% edge-addition / 30% edge-removal** update process;
- gives every baseline its **own sequential coloring state**;
- aligns fresh global color labels before counting channel changes, so harmless label permutations are not counted as reassignments.

Run the cells from top to bottom. Training can take a while on CPU.

In [ ]:
!pip install -q torch torch_geometric networkx numpy scipy tqdm

In [ ]:
# Colab opened directly from GitHub does not always clone the rest of the repo.
import os
if not os.path.exists('dynamic_gnn.py'):
    !wget -q https://raw.githubusercontent.com/YaolaP4/Incremental_GNN_For_Combinatorial_Problems/main/dynamic_gnn.py

import torch
from dynamic_gnn import (
    ConflictAwareDynamicGNN,
    DynamicGraphProcessor,
    benchmark_suite,
    count_conflicts,
    format_summary,
    generate_heldout_graphs,
    generate_training_graphs,
    graph_to_edge_index,
    pick_device,
    post_process_coloring,
    set_seed,
    train_incremental_model,
    train_static_model,
)

set_seed(42)
DEVICE = pick_device()
MAX_COLORS = 50
print('Using device:', DEVICE)

## Sanity-check conflict handling

In [ ]:
import networkx as nx

# This coloring has exactly two conflicts: (1,2) and (3,4).
check_graph = nx.path_graph(5)
check_edges = graph_to_edge_index(check_graph, DEVICE)
check_colors = torch.tensor([0, 1, 1, 0, 0], device=DEVICE)
assert count_conflicts(check_edges, check_colors) == 2
repaired = post_process_coloring(check_colors, check_edges, max_colors=5)
assert count_conflicts(check_edges, repaired) == 0
print('Conflict counting and repair sanity check passed.')

## Build training and held-out evaluation graphs

In [ ]:
train_graphs = generate_training_graphs(seed=1000)
heldout = generate_heldout_graphs(seed=9000)

print('Training graphs:', len(train_graphs))
for name, graphs in heldout.items():
    print(f'{name}: {len(graphs)} held-out graphs')

## Train the model

In [ ]:
processor = DynamicGraphProcessor(max_colors=MAX_COLORS, device=DEVICE)
model = ConflictAwareDynamicGNN(max_colors=MAX_COLORS, hidden=64, gat_heads=2).to(DEVICE)

# Two-phase schedule: static pre-training, then incremental training.
model = train_static_model(
    model,
    processor,
    train_graphs,
    epochs=20,
)

model = train_incremental_model(
    model,
    processor,
    train_graphs,
    epochs=30,
    seed=2000,
)

torch.save(model.state_dict(), 'incremental_gnn_corrected.pt')
print('Saved incremental_gnn_corrected.pt')

## Held-out dynamic evaluation

In [ ]:
results = benchmark_suite(
    model,
    heldout,
    device=DEVICE,
    max_colors=MAX_COLORS,
    rounds=10,
    seed=7000,
)

for category, payload in results.items():
    print('\n' + '=' * 72)
    print(category.upper())
    print(format_summary(payload['summary']))

### Reading the results

`Conflicts` is measured over **all unique undirected edges** after each update. The benchmark raises an exception if any evaluated method returns an invalid coloring, so a displayed mean of `0.00` is validated rather than hard-coded.

`Changes` is measured against each method's **own previous coloring**. DSATUR and Welsh-Powell are global recoloring methods, so their newly generated color IDs are first aligned to the previous channel labels by maximum overlap. This avoids counting a pure permutation of color names as a network disruption.

The held-out sets are created independently from the training graphs. The `realistic` set uses random-geometric graphs as a spatial proxy; it is still synthetic and should not be described as a field deployment.